# NOTES
* PyTorch's Flash Attention implementation requires Ampere or Blackwell chips (SM80 or SM121 architecture)

# Test Flash Attention

In [0]:
# %pip install -U torch
# %restart_python

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [0]:
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_math_sdp(False)

In [0]:
query = torch.randn(2, 3, 8, 1, device=device, dtype=torch.bfloat16)
key = torch.randn(2, 3, 8, 1, device=device, dtype=torch.bfloat16)
value = torch.randn(2, 3, 8, 1, device=device, dtype=torch.bfloat16)

In [0]:
F.scaled_dot_product_attention(query, key, value)

# Data Preparation

## Setup database and volume

In [0]:
# dbutils.fs.rm(f'{ROOT_PATH}/data', True)
# spark.sql('drop table if exists vr_demo.dl.fineweb_tokens')

In [0]:
%sql create database if not exists vr_demo.dl

In [0]:
%sql create volume if not exists vr_demo.dl.volume

## Download raw files

In [0]:
%pip install datatrove

In [0]:
from datatrove.pipeline.readers import ParquetReader
from datatrove.pipeline.writers import ParquetWriter
from datatrove.executor import LocalPipelineExecutor

pipeline = [
    ParquetReader("hf://datasets/HuggingFaceFW/fineweb-edu/sample/10BT", limit=64*8*2),
    ParquetWriter(RAW_DATA_PATH)
]

executor = LocalPipelineExecutor(pipeline=pipeline, tasks=1)
executor.run()

## Tokenize and chunk documents

In [0]:
from pyspark.sql.functions import udf, col, explode
from pyspark.sql.types import ArrayType, IntegerType
import numpy as np
import tiktoken

BLOCK_SIZE = 1024

enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>'] # end of text token

def tokenize_and_chunk(doc):
    tokens = [eot] # the special <|endoftext|> token delimits all documents
    tokens.extend(enc.encode_ordinary(doc))
    if len(tokens) < BLOCK_SIZE+1: # adds padding if sequence is too short
        tokens.extend([eot] * (BLOCK_SIZE+1 - len(tokens)))
    return [tokens[i:i + BLOCK_SIZE+1] if i + BLOCK_SIZE+1 <= len(tokens) else tokens[-BLOCK_SIZE-1:] for i in range(0, len(tokens), BLOCK_SIZE)]

tokenize_udf = udf(tokenize_and_chunk, ArrayType(ArrayType(IntegerType())))

(spark.read.format('parquet').load(RAW_DATA_PATH) \
    .select(explode(tokenize_udf(col("text"))).alias('tokens'))
    .write.mode('overwrite').saveAsTable('vr_demo.dl.fineweb_tokens'))

In [0]:
display(spark.table('vr_demo.dl.fineweb_tokens'))

## Convert to MDS format

In [0]:
from streaming.base.converters import dataframe_to_mds

df = spark.read.table('vr_demo.dl.fineweb_tokens')
df_train, df_val, df_test = df.randomSplit([0.8, 0.1, 0.1])

dataframe_to_mds(df_train, merge_index=True, mds_kwargs={'out': f'{MDS_DATA_PATH}/train', 'columns': {'tokens': 'ndarray:int32'}})
dataframe_to_mds(df_val, merge_index=True, mds_kwargs={'out': f'{MDS_DATA_PATH}/val', 'columns': {'tokens': 'ndarray:int32'}})
dataframe_to_mds(df_test, merge_index=True, mds_kwargs={'out': f'{MDS_DATA_PATH}/test', 'columns': {'tokens': 'ndarray:int32'}})

# GPT2 + Flash Attention

## Dependencies

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor
import mlflow
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
import math
import inspect
import os

## Parameters

In [0]:
# Torch Distributor
NUM_PROCESSES = 8
LOCAL_MODE = True

# Training
EPOCH_SIZE = 1024
BATCH_SIZE = 16
NUM_EPOCHS = 1 # 64 steps

# Paths
ROOT_PATH = '/Volumes/vr_demo/dl/volume'
RAW_DATA_PATH = f'{ROOT_PATH}/data/raw'
MDS_DATA_PATH = f'{ROOT_PATH}/data/mds'
LOCAL_PATH = "/tmp/fineweb/mds"
CHECKPOINT_PATH = f'{ROOT_PATH}/checkpoints'

# MLflow 
DB_HOST = 'https://e2-demo-field-eng.cloud.databricks.com'
DB_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
EXPERIMENT_PATH = '/Users/victor.rodrigues@databricks.com/torch-distributor-gpt2-flash-attn'
LOG_INTERVAL = 1000

# Learninig rate warm-up
MAX_LR = 6e-4
MIN_LR = MAX_LR * 0.1
WARMUP_STEPS = 715
MAX_STEPS = 19073 # 19,073 steps is ~1 epoch, if data is 10B tokens and batch size 0.5M tokens

## Data Loader

In [0]:
from streaming import StreamingDataset, StreamingDataLoader
from streaming.base.util import clean_stale_shared_memory

def get_dataloader_with_mosaic(remote_path, local_path, batch_size):
  print(f"Getting data from UC Volumes at {remote_path}")

  # Utility function to clean up stale shared memory during distributed training
  clean_stale_shared_memory()

  # Creating the `StreamingDataset` object and the `StreamingDataLoader` object.
  dataset = StreamingDataset(remote=remote_path, local=local_path, shuffle=True, batch_size=batch_size, epoch_size=EPOCH_SIZE)
  return StreamingDataLoader(dataset, batch_size=batch_size)

In [0]:
# Test training Data Loader
train_dataloader = get_dataloader_with_mosaic(
    remote_path = f'{MDS_DATA_PATH}/train',
    local_path = f'{LOCAL_PATH}/2train',
    batch_size = BATCH_SIZE)
train_steps_per_epoch = len(train_dataloader)
print(train_steps_per_epoch)

In [0]:
t = 0
for i, data in enumerate(train_dataloader):
    print(data['tokens'].shape)
    t += data['tokens'].shape[0]
print(t)

## GPTConfig

In [0]:
@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    n_layer: int = 12 # number of layers
    n_head: int = 12 # number of heads
    n_embd: int = 768 # embedding dimension

## Flash Attention

In [0]:
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_math_sdp(False)

In [0]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh is "number of heads", hs is "head size", and C (number of channels) = nh * hs
        # e.g. in GPT-2 (124M), n_head=12, hs=64, so nh*hs=C=768 channels in the Transformer
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True) # flash attention
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

## MLP

In [0]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

## Block

In [0]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## GPT

In [0]:
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight sharing scheme
        self.transformer.wte.weight = self.lm_head.weight

        # init params
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        # forward the token and posisition embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb
        # forward the blocks of the transformer
        for block in self.transformer.h:
            x = block(x)
        # forward the final layernorm and the classifier
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

## Optimizer

In [0]:
def configure_optimizer(model, weight_decay, learning_rate, device_type):
    # start with all of the candidate parameters (that require grad)
    param_dict = {pn: p for pn, p in model.module.named_parameters()}
    param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
    # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
    print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
    # Create AdamW optimizer and use the fused version if it is available
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device_type == "cuda"
    print(f"using fused AdamW: {use_fused}")
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
    return optimizer

## Training functions

#### create_log_dir

In [0]:
from time import time

def create_log_dir():
  log_dir = os.path.join(CHECKPOINT_PATH, str(time()))
  os.makedirs(log_dir)
  return log_dir

#### get_lr

In [0]:
def get_lr(it):
    # 1) linear warmup for warmup_iters steps
    if it < WARMUP_STEPS:
        return MAX_LR * (it+1) / WARMUP_STEPS
    # 2) if it > lr_decay_iters, return min learning rate
    if it > MAX_STEPS:
        return MIN_LR
    # 3) in between, use cosine decay down to min learning rate
    decay_ratio = (it - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff starts at 1 and goes to 0
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

#### train_one_epoch

In [0]:
def train_one_epoch(model, device, data_loader, optimizer, epoch):
  model.train()
  # for batch_idx, (data, target) in enumerate(data_loader):
  for batch_idx, i in enumerate(data_loader):
    data = i['tokens'][:, :-1]
    target = i['tokens'][:, 1:].type(torch.LongTensor)
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output, loss = model(data, target)
    # TODO: SCALE LOSS
    # loss = F.cross_entropy(output, target)
    loss.backward()
    # Added gradient clipping
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Added learning rate warmup
    lr = get_lr(batch_idx)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    optimizer.step()
    if batch_idx % LOG_INTERVAL == 0:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
          epoch, batch_idx * len(data), len(data_loader) * len(data),
          100. * batch_idx / len(data_loader), loss.item()))
      
      if int(os.environ["RANK"]) == 0:
        mlflow.log_metric('train_loss', loss.item())

#### save_checkpoint

In [0]:
def save_checkpoint(log_dir, model, optimizer, epoch):
  filepath = log_dir + '/checkpoint-{epoch}.pth.tar'.format(epoch=epoch)
  state = {
    'model': model.module.state_dict(),
    'optimizer': optimizer.state_dict(),
  }
  torch.save(state, filepath)

#### main_fn

In [0]:
# For distributed training we will merge the train and test steps into 1 main function
def main_fn(directory):
  
  #### Added imports here ####
  import mlflow
  import torch.distributed as dist
  from torch.nn.parallel import DistributedDataParallel as DDP
  from torch.utils.data.distributed import DistributedSampler
  ############################

  ##### Setting up MLflow ####
  # We need to do this so that different processes that will be able to find mlflow
  os.environ['DATABRICKS_HOST'] = DB_HOST
  os.environ['DATABRICKS_TOKEN'] = DB_TOKEN

  # We set the experiment details here
  experiment = mlflow.set_experiment(EXPERIMENT_PATH)
  ############################
  
  print("Running distributed training")
  dist.init_process_group("nccl")
  
  local_rank = int(os.environ["LOCAL_RANK"])
  global_rank = int(os.environ["RANK"])
  
  # Log parameters
  if global_rank == 0:
    train_parameters = {'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'trainer': 'TorchDistributor'}
    mlflow.log_params(train_parameters)
  
  #### Added Distributed Dataloader ####
  # train_sampler = DistributedSampler(dataset=train_dataset)
  # data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
  train_dataloader = get_dataloader_with_mosaic(
    remote_path = f'{MDS_DATA_PATH}/train',
    local_path = f'{LOCAL_PATH}/train',
    batch_size = BATCH_SIZE
  )
  ######################################

  # Initialize model from scratch or pretrained
  # model = GPT.from_pretrained("gpt2")
  model = GPT(GPTConfig(vocab_size=50304))
  model.to(local_rank)
  #### Added Distributed Model ####
  ddp_model = DDP(model, device_ids=[local_rank], output_device=local_rank)
  #################################
  
  # Initialize optimizer
  optimizer = configure_optimizer(model=ddp_model, weight_decay=0.1, learning_rate=6e-4, device_type="cuda")

  # Train model
  for epoch in range(1, NUM_EPOCHS + 1):
    train_one_epoch(ddp_model, local_rank, train_dataloader, optimizer, epoch)
    # Save checkpoint
    if global_rank == 0: 
      save_checkpoint(directory, ddp_model, optimizer, epoch)
  
  # Save and evaluate model
  if global_rank == 0:
    # Save model
    mlflow.pytorch.log_model(ddp_model, "model")

    # Create training Data Loader
    val_dataloader = get_dataloader_with_mosaic(
      remote_path = f'{MDS_DATA_PATH}/val',
      local_path = f'{LOCAL_PATH}/val',
      batch_size = BATCH_SIZE
    )
    
    # Evaluate and log model metrics
    # ddp_model.eval()
    test_loss = 0
    for data, target in val_dataloader:
      device = torch.device('cuda')
      data, target = data.to(device), target.to(device)
      output, loss = ddp_model(data)
      test_loss += F.cross_entropy(output, target)
    test_loss /= len(val_dataloader)
    mlflow.log_metric('test_loss', test_loss.item())
    print("Average test loss: {}".format(test_loss.item()))
    
  # Cleanup the distributed environment
  dist.destroy_process_group()
  
  return "finished" # can return any picklable object

## Run training

In [0]:
checkpoint_path = create_log_dir()
print("Data is located at: ", checkpoint_path)

experiment = mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run():
    output_dist = TorchDistributor(num_processes=NUM_PROCESSES, local_mode=LOCAL_MODE, use_gpu=True).run(main_fn, checkpoint_path)